In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
import pandas as pd
from trino.dbapi import connect
from trino.auth import BasicAuthentication
import datetime
import os

In [3]:
trino_catalog = "hive"
trino_schema = "bi_silver"
# trino_schema = "bi_gold"

# --- 1. CẤU HÌNH HỆ THỐNG ---
CONFIG = {
    'host': os.getenv("VCS_TRINO_HOST"),
    'port': int(os.getenv("VCS_TRINO_PORT")),
    'user': os.getenv("VCS_TRINO_USER"),
    'catalog': trino_catalog,
    'schema': trino_schema,  # Schema mục tiêu
    'password': os.getenv("VCS_TRINO_PASSWORD"),
}


def get_connection():
    return connect(
        host=CONFIG['host'],
        port=CONFIG['port'],
        user=CONFIG['user'],
        catalog=CONFIG['catalog'],
        http_scheme='https',
        auth=BasicAuthentication(CONFIG['user'], CONFIG['password']),
        verify=False,
    )
    

In [4]:
# --- 2. BUILD SQL DQ (FIXED FOR PARQUET METADATA) ---
def build_dq_query(table_path, columns_df):
    # Sử dụng TRY_CAST ngay cả trong lệnh COUNT để tránh lỗi Unsupported Type từ Parquet
    sql_parts = ["COUNT(*) as total_rows"]
    
    for idx, row in columns_df.iterrows():
        raw_col = row['Column']
        quoted_col = f"\"{raw_col}\""
        dtype = row['Type'].lower()
        p = f"c{idx}" 
        
        # FIX: Dùng TRY_CAST cho mọi lệnh COUNT để an toàn tuyệt đối với Parquet
        safe_count_col = f"TRY_CAST({quoted_col} AS VARCHAR)" if 'timestamp' in dtype else quoted_col
        
        sql_parts.append(f"""
                         CASE
        WHEN COUNT(DISTINCT {safe_count_col}) < 50
            THEN array_join(array_agg(DISTINCT CAST({safe_count_col} AS VARCHAR)), ', ')
        ELSE NULL
    END AS {p}_result_text
                         """)
        sql_parts.append(f"COUNT({safe_count_col}) AS {p}_nonnull")
        sql_parts.append(f"APPROX_DISTINCT({safe_count_col}) AS {p}_unique")
        
        if 'varchar' in dtype or 'char' in dtype:
            safe_col = f"TRY_CAST({quoted_col} AS VARCHAR)"
            sql_parts.append(f"SUM(CASE WHEN {safe_col} != TRIM({safe_col}) THEN 1 ELSE 0 END) AS {p}_ws")
            sql_parts.append(f"SUM(CASE WHEN {safe_col} = '' THEN 1 ELSE 0 END) AS {p}_empty")
            
            if not any(x in raw_col.lower() for x in ['id', 'code', 'email', 'url', 'key', 'timestamp']):
                sql_parts.append(f"SUM(CASE WHEN REGEXP_LIKE({safe_col}, '[0-9!@#$%^&*()]') THEN 1 ELSE 0 END) AS {p}_worderr")
            else:
                sql_parts.append(f"0 AS {p}_worderr")
                
            sql_parts.append(f"SUM(CASE WHEN TRY_CAST({safe_col} AS DOUBLE) IS NOT NULL AND {safe_col} != '' THEN 1 ELSE 0 END) AS {p}_isnum")
            sql_parts.append(f"SUM(CASE WHEN TRY_CAST({safe_col} AS TIMESTAMP) IS NOT NULL THEN 1 ELSE 0 END) AS {p}_isdate")
            sql_parts.append(f"SUM(CASE WHEN REGEXP_LIKE({safe_col}, '^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{{2,}}$') THEN 1 ELSE 0 END) AS {p}_isemail")

        elif any(t in dtype for t in ['int', 'decimal', 'double', 'real', 'bigint']):
            sql_parts.append(f"SUM(CASE WHEN {quoted_col} < 0 THEN 1 ELSE 0 END) AS {p}_neg")
            sql_parts.append(f"APPROX_PERCENTILE({quoted_col}, 0.25) AS {p}_q25")
            sql_parts.append(f"APPROX_PERCENTILE({quoted_col}, 0.50) AS {p}_q50")
            sql_parts.append(f"APPROX_PERCENTILE({quoted_col}, 0.75) AS {p}_q75")
            sql_parts.append(f"APPROX_PERCENTILE({quoted_col}, 0.95) AS {p}_q95")
        else:
            # Case an toàn cho Timestamp bị lỗi định dạng file
            sql_parts.extend([f"0 AS {p}_ws", f"0 AS {p}_empty", f"0 AS {p}_worderr", f"0 AS {p}_isnum", f"0 AS {p}_isdate", f"0 AS {p}_isemail"])

    return f"SELECT {', '.join(sql_parts)} FROM {table_path}"

# --- 3. RUN AUDIT ---
def run_data_quality_audit(output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Format thời gian đầy đủ YYYYMMDD_HHMMSS
    time_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = os.path.join(output_dir, f"DQ_Master_Report_{time_str}.xlsx")

    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT table_name FROM hive.information_schema.tables WHERE table_schema = '{trino_schema}'")
    tables = [row[0] for row in cursor.fetchall()]
    
    summary_data = []

    with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
        for table_name in tables:
            print(f"--> Analyzing table: {table_name}")
            full_path = f"{trino_catalog}.{trino_schema}.\"{table_name}\""
            
            try:
                cols_df = pd.read_sql_query(f"SHOW COLUMNS FROM {full_path}", conn)
                dq_sql = build_dq_query(full_path, cols_df)
                # print("=================dq_sql\n\n")
                # print(dq_sql)
                # print("=================dq_sql\n\n")
                res = pd.read_sql_query(dq_sql, conn)
                
                total = res['total_rows'][0]
                table_details = []
                for i, row in cols_df.iterrows():
                    p = f"c{i}"
                    non_null = res[f"{p}_nonnull"][0]
                    
                    is_num = res.get(f"{p}_isnum", [0])[0]
                    is_date = res.get(f"{p}_isdate", [0])[0]
                    suggestion = "Giữ nguyên"
                    if total > 0 and non_null > 0 and 'varchar' in row['Type'].lower():
                        if is_num == non_null: suggestion = "Nên đổi sang NUMBER"
                        elif is_date == non_null: suggestion = "Nên đổi sang DATE/TIMESTAMP"

                    table_details.append({
                        'Cột': row['Column'], 'Kiểu': row['Type'], 'Null': total - non_null,
                        'Khoảng trắng': res.get(f"{p}_ws", [0])[0], 'Dòng rỗng': res.get(f"{p}_empty", [0])[0],
                        'Ký tự lạ': res.get(f"{p}_worderr", [0])[0], 'Email hợp lệ': res.get(f"{p}_isemail", [0])[0],
                        'Lỗi âm': res.get(f"{p}_neg", [0])[0], 'Q25': res.get(f"{p}_q25", [None])[0],
                        'Q50': res.get(f"{p}_q50", [None])[0], 'Q75': res.get(f"{p}_q75", [None])[0],
                        'Q95': res.get(f"{p}_q95", [None])[0], 'GỢI Ý': suggestion, 
                        'enums': res.get(f"{p}_result_text", [None])[0],
                    })
                
                sheet_name = table_name[:31]
                pd.DataFrame(table_details).to_excel(writer, sheet_name=sheet_name, index=False)

                # Format cột: Tăng độ rộng thêm 50%
                ws = writer.sheets[sheet_name]
                for col in ws.columns:
                    max_length = 0
                    for cell in col:
                        try:
                            if len(str(cell.value)) > max_length: max_length = len(str(cell.value))
                        except: pass
                    ws.column_dimensions[col[0].column_letter].width = (max_length + 2) * 1.5

                summary_data.append({'Bảng': table_name, 'Dòng': total, 'Trạng thái': 'OK'})
                
            except Exception as e:
                print(f"❌ Error at {table_name}: {e}")

        if summary_data:
            pd.DataFrame(summary_data).to_excel(writer, sheet_name='DASHBOARD', index=False)

    print(f"\n✅ Done! File: {file_path}")

In [5]:
run_data_quality_audit(r"C:\Users\namtv40\Data\DQ-Bitu-Silver-Data")

c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namt

--> Analyzing table: hr_ld_budget_detail


C:\Users\namtv40\AppData\Local\Temp\ipykernel_15032\3347723271.py:73: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cols_df = pd.read_sql_query(f"SHOW COLUMNS FROM {full_path}", conn)
c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c

--> Analyzing table: noc_entities


c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namt

...

❌ Error at cx_lifecycle_customer_contract_snapshot: Execution failed on sql: SELECT COUNT(*) as total_rows, 
                         CASE
        WHEN COUNT(DISTINCT "report_date") < 50
            THEN array_join(array_agg(DISTINCT CAST("report_date" AS VARCHAR)), ', ')
        ELSE NULL
    END AS c0_result_text
                         , COUNT("report_date") AS c0_nonnull, APPROX_DISTINCT("report_date") AS c0_unique, 0 AS c0_ws, 0 AS c0_empty, 0 AS c0_worderr, 0 AS c0_isnum, 0 AS c0_isdate, 0 AS c0_isemail, 
                         CASE
        WHEN COUNT(DISTINCT "tax_code") < 50
            THEN array_join(array_agg(DISTINCT CAST("tax_code" AS VARCHAR)), ', ')
        ELSE NULL
    END AS c1_result_text
                         , COUNT("tax_code") AS c1_nonnull, APPROX_DISTINCT("tax_code") AS c1_unique, SUM(CASE WHEN TRY_CAST("tax_code" AS VARCHAR) != TRIM(TRY_CAST("tax_code" AS VARCHAR)) THEN 1 ELSE 0 END) AS c1_ws, SUM(CASE WHEN TRY_CAST("tax_code" AS VARCHAR) = '' THEN 1 ELSE

c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


❌ Error at finance_allocated_revenue: Execution failed on sql: SHOW COLUMNS FROM hive.bi_silver."finance_allocated_revenue"
error 500: b'java.lang.IllegalStateException: authenticators were not loaded\n\tat com.google.common.base.Preconditions.checkState(Preconditions.java:513)\n\tat io.trino.server.security.PasswordAuthenticatorManager.getAuthenticators(PasswordAuthenticatorManager.java:120)\n\tat io.trino.server.security.PasswordAuthenticator.authenticate(PasswordAuthenticator.java:61)\n\tat io.trino.server.security.AuthenticationFilter.filter(AuthenticationFilter.java:87)\n\tat org.glassfish.jersey.server.ContainerFilteringStage.apply(ContainerFilteringStage.java:108)\n\tat org.glassfish.jersey.server.ContainerFilteringStage.apply(ContainerFilteringStage.java:44)\n\tat org.glassfish.jersey.process.internal.Stages.process(Stages.java:173)\n\tat org.glassfish.jersey.server.ServerRuntime$1.run(ServerRuntime.java:266)\n\tat org.glassfish.jersey.internal.Errors$1.call(Errors.java:248)\n\

c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namtv40\AppData\Local\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.255.245.150'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\namt

❌ Error at crm_expected_revenue: Execution failed on sql: SHOW COLUMNS FROM hive.bi_silver."crm_expected_revenue"
error 500: b'java.lang.IllegalStateException: authenticators were not loaded\n\tat com.google.common.base.Preconditions.checkState(Preconditions.java:513)\n\tat io.trino.server.security.PasswordAuthenticatorManager.getAuthenticators(PasswordAuthenticatorManager.java:120)\n\tat io.trino.server.security.PasswordAuthenticator.authenticate(PasswordAuthenticator.java:61)\n\tat io.trino.server.security.AuthenticationFilter.filter(AuthenticationFilter.java:87)\n\tat org.glassfish.jersey.server.ContainerFilteringStage.apply(ContainerFilteringStage.java:108)\n\tat org.glassfish.jersey.server.ContainerFilteringStage.apply(ContainerFilteringStage.java:44)\n\tat org.glassfish.jersey.process.internal.Stages.process(Stages.java:173)\n\tat org.glassfish.jersey.server.ServerRuntime$1.run(ServerRuntime.java:266)\n\tat org.glassfish.jersey.internal.Errors$1.call(Errors.java:248)\n\tat org.gl